<a href="https://colab.research.google.com/github/Tejas-Acharya-C/api-data-engineering-pipeline/blob/main/api_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# API → Data Engineering Pipeline

A hands-on data engineering project covering:

- API data extraction
- JSON processing
- Data cleaning
- Data validation
- Parquet
- DuckDB
- SQL analytics
- Incremental processing
- Python pipeline development

## 1. Extract Data from API

We will make an HTTP GET request to a public API and inspect the JSON response.

In [51]:
import requests

In [52]:
url = "https://jsonplaceholder.typicode.com/users"

In [53]:
response = requests.get(url)

In [54]:
print(response.status_code)

200


In [55]:
data = response.json()

In [56]:
print(type(data))

<class 'list'>


In [57]:
print(len(data))

10


In [58]:
print(data[0])

{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}


In [59]:
print(response.status_code)
print(type(data))
print(len(data))
print(data[0])

200
<class 'list'>
10
{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}


## 2. Store Raw API Data

We save the original API response before performing any transformations.

In [60]:
import os

os.makedirs("data/raw", exist_ok=True)

In [61]:
import json

with open("data/raw/users.json", "w") as f:
    json.dump(data, f, indent=2)

In [62]:
os.listdir("data/raw")

['users.json']

## 3. Convert JSON to DataFrame

Convert the nested JSON response into a tabular structure using Pandas.

In [63]:
import pandas as pd

In [64]:
df = pd.DataFrame(data)

In [65]:
df.head()

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


In [66]:
df = pd.json_normalize(data)

In [67]:
df.head()

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


In [68]:
df.columns.tolist()

['id',
 'name',
 'username',
 'email',
 'phone',
 'website',
 'address.street',
 'address.suite',
 'address.city',
 'address.zipcode',
 'address.geo.lat',
 'address.geo.lng',
 'company.name',
 'company.catchPhrase',
 'company.bs']

In [69]:
df.shape

(10, 15)

In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   10 non-null     int64 
 1   name                 10 non-null     object
 2   username             10 non-null     object
 3   email                10 non-null     object
 4   phone                10 non-null     object
 5   website              10 non-null     object
 6   address.street       10 non-null     object
 7   address.suite        10 non-null     object
 8   address.city         10 non-null     object
 9   address.zipcode      10 non-null     object
 10  address.geo.lat      10 non-null     object
 11  address.geo.lng      10 non-null     object
 12  company.name         10 non-null     object
 13  company.catchPhrase  10 non-null     object
 14  company.bs           10 non-null     object
dtypes: int64(1), object(14)
memory usage: 1.3+ KB


In [71]:
df.head(3)

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications


In [72]:
df.tail(3)

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,586.493.6943 x140,jacynthe.com,Ellsworth Summit,Suite 729,Aliyaview,45169,-14.3990,-120.7677,Abernathy Group,Implemented secondary concept,e-enable extensible e-tailers
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,(775)976-6794 x41206,conrad.com,Dayna Park,Suite 449,Bartholomebury,76495-3109,24.6463,-168.8889,Yost and Sons,Switchable contextually-based project,aggregate real-time technologies
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,024-648-3804,ambrose.net,Kattie Turnpike,Suite 198,Lebsackbury,31428-2261,-38.2386,57.2232,Hoeger LLC,Centralized empowering task-force,target end-to-end models


In [73]:
df.shape
df.columns.tolist()
df.head(3)

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications


In [74]:
df.shape

(10, 15)

In [75]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   10 non-null     int64 
 1   name                 10 non-null     object
 2   username             10 non-null     object
 3   email                10 non-null     object
 4   phone                10 non-null     object
 5   website              10 non-null     object
 6   address.street       10 non-null     object
 7   address.suite        10 non-null     object
 8   address.city         10 non-null     object
 9   address.zipcode      10 non-null     object
 10  address.geo.lat      10 non-null     object
 11  address.geo.lng      10 non-null     object
 12  company.name         10 non-null     object
 13  company.catchPhrase  10 non-null     object
 14  company.bs           10 non-null     object
dtypes: int64(1), object(14)
memory usage: 1.3+ KB


In [76]:
df.isnull().sum()

,0
id,0
name,0
username,0
email,0
phone,0
website,0
address.street,0
address.suite,0
address.city,0
address.zipcode,0


In [77]:
df.duplicated().sum()

np.int64(0)

In [78]:
df["id"].duplicated().sum()

np.int64(0)

In [79]:
df["id"].is_unique

True

In [80]:
df.nunique()

,0
id,10
name,10
username,10
email,10
phone,10
website,10
address.street,10
address.suite,10
address.city,10
address.zipcode,10


In [81]:
df[["address.geo.lat", "address.geo.lng"]].head()

,address.geo.lat,address.geo.lng
0,-37.3159,81.1496
1,-43.9509,-34.4618
2,-68.6102,-47.0653
3,29.4572,-164.2990
4,-31.8129,62.5342


In [82]:
df[["address.geo.lat", "address.geo.lng"]].dtypes

,0
address.geo.lat,object
address.geo.lng,object


In [83]:
df.shape
df.info()
df.isnull().sum()
df.duplicated().sum()
df["id"].duplicated().sum()
df[["address.geo.lat", "address.geo.lng"]].dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   10 non-null     int64 
 1   name                 10 non-null     object
 2   username             10 non-null     object
 3   email                10 non-null     object
 4   phone                10 non-null     object
 5   website              10 non-null     object
 6   address.street       10 non-null     object
 7   address.suite        10 non-null     object
 8   address.city         10 non-null     object
 9   address.zipcode      10 non-null     object
 10  address.geo.lat      10 non-null     object
 11  address.geo.lng      10 non-null     object
 12  company.name         10 non-null     object
 13  company.catchPhrase  10 non-null     object
 14  company.bs           10 non-null     object
dtypes: int64(1), object(14)
memory usage: 1.3+ KB


,0
address.geo.lat,object
address.geo.lng,object


In [84]:
df = df.rename(columns={
    "address.street": "street",
    "address.suite": "suite",
    "address.city": "city",
    "address.zipcode": "zipcode",
    "address.geo.lat": "latitude",
    "address.geo.lng": "longitude",
    "company.name": "company_name",
    "company.catchPhrase": "company_catchphrase",
    "company.bs": "company_bs"
})

In [85]:
df.columns.tolist()

['id',
 'name',
 'username',
 'email',
 'phone',
 'website',
 'street',
 'suite',
 'city',
 'zipcode',
 'latitude',
 'longitude',
 'company_name',
 'company_catchphrase',
 'company_bs']

In [86]:
df[["latitude", "longitude"]].dtypes

,0
latitude,object
longitude,object


In [87]:
df["latitude"] = pd.to_numeric(df["latitude"])
df["longitude"] = pd.to_numeric(df["longitude"])

In [88]:
df[["latitude", "longitude"]].dtypes

,0
latitude,float64
longitude,float64


In [89]:
df[["latitude", "longitude"]].head()

,latitude,longitude
0,-37.3159,81.1496
1,-43.9509,-34.4618
2,-68.6102,-47.0653
3,29.4572,-164.2990
4,-31.8129,62.5342


In [90]:
df[["latitude", "longitude"]].dtypes

,0
latitude,float64
longitude,float64


In [91]:
df.columns.tolist()
df[["latitude", "longitude"]].dtypes

,0
latitude,float64
longitude,float64


In [92]:
df["latitude"].min(), df["latitude"].max()

(-71.4197, 29.4572)

In [93]:
df["longitude"].min(), df["longitude"].max()

(-168.8889, 81.1496)

In [94]:
assert df["latitude"].between(-90, 90).all()
assert df["longitude"].between(-180, 180).all()

In [95]:
assert df["latitude"].between(-90, 90).all()

In [96]:
def validate_users(df):
    assert df["id"].notna().all(), "Missing user IDs"
    assert df["id"].is_unique, "Duplicate user IDs"

    assert df["email"].notna().all(), "Missing emails"

    assert df["latitude"].between(-90, 90).all(), "Invalid latitude"
    assert df["longitude"].between(-180, 180).all(), "Invalid longitude"

    return True

In [97]:
validate_users(df)

True

In [98]:
test_df = df.copy()

In [99]:
test_df.loc[0, "latitude"] = 200

In [100]:
validate_users(test_df)

AssertionError: Invalid latitude

In [ ]:
validate_users(df)

In [ ]:
os.makedirs("data/processed", exist_ok=True)

In [ ]:
os.listdir("data")

In [ ]:
df.to_parquet(
    "data/processed/users.parquet",
    index=False
)

In [ ]:
os.listdir("data/processed")

In [ ]:
processed_df = pd.read_parquet(
    "data/processed/users.parquet"
)

In [ ]:
processed_df.head()

In [ ]:
import os

json_size = os.path.getsize("data/raw/users.json")
parquet_size = os.path.getsize("data/processed/users.parquet")

print("JSON:", json_size, "bytes")
print("Parquet:", parquet_size, "bytes")

In [ ]:
processed_df.shape

In [101]:
processed_df.dtypes

NameError: name 'processed_df' is not defined

In [ ]:
validate_users(processed_df)

In [ ]:
os.makedirs("data/processed", exist_ok=True)

In [ ]:
df.to_parquet(
    "data/processed/users.parquet",
    index=False
)

In [ ]:
processed_df = pd.read_parquet(
    "data/processed/users.parquet"
)

In [102]:
processed_df.shape
processed_df.dtypes
validate_users(processed_df)

NameError: name 'processed_df' is not defined

In [ ]:
!pip install -q duckdb

In [103]:
import duckdb

In [104]:
con = duckdb.connect()

In [105]:
result = con.execute("""
    SELECT *
    FROM 'data/processed/users.parquet'
""").fetchdf()

IOException: IO Error: No files found that match the pattern "data/processed/users.parquet"

In [ ]:
result.head()

In [ ]:
con.execute("""
    SELECT
        id,
        name,
        city
    FROM 'data/processed/users.parquet'
""").fetchdf()

In [ ]:
con.execute("""
    SELECT
        id,
        name,
        city
    FROM 'data/processed/users.parquet'
    WHERE city = 'Gwenborough'
""").fetchdf()

In [ ]:
con.execute("""
    SELECT
        id,
        name,
        city
    FROM 'data/processed/users.parquet'
    ORDER BY name
""").fetchdf()

In [ ]:
con.execute("""
    SELECT COUNT(*) AS total_users
    FROM 'data/processed/users.parquet'
""").fetchdf()

In [ ]:
con.execute("""
    SELECT
        city,
        COUNT(*) AS user_count
    FROM 'data/processed/users.parquet'
    GROUP BY city
""").fetchdf()

In [ ]:
con.execute("""
    SELECT
        city,
        COUNT(*) AS user_count
    FROM 'data/processed/users.parquet'
    GROUP BY city
    ORDER BY user_count DESC
""").fetchdf()

In [ ]:
con.execute("""
    SELECT
        company_name,
        COUNT(*) AS user_count
    FROM 'data/processed/users.parquet'
    GROUP BY company_name
    ORDER BY user_count DESC
""").fetchdf()

In [ ]:
con.execute("""
    SELECT
        company_name,
        COUNT(*) AS user_count
    FROM 'data/processed/users.parquet'
    GROUP BY company_name
    ORDER BY user_count DESC
""").fetchdf()

In [ ]:
con.execute("""
    SELECT
        COUNT(*) AS user_count
    FROM 'data/processed/users.parquet'
    WHERE NAME LIKE '%a%'
""").fetchdf()

In [ ]:
con.execute("""
    SELECT
        name,email
    FROM 'data/processed/users.parquet'
    WHERE CITY LIKE '%view%'
""").fetchdf()

In [106]:
import pandas as pd

processed_df = pd.read_parquet("data/processed/users.parquet")

processed_df.dtypes

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/users.parquet'

In [107]:
import os

print(os.listdir("."))

['.config', 'data', 'sample_data']


In [108]:
import os

for root, dirs, files in os.walk("data"):
    print(root, "->", files)

data -> []
data/raw -> ['users.json']


In [109]:
import pandas as pd

df = pd.read_json("data/raw/users.json")
df = pd.json_normalize(df.to_dict(orient="records"))

print(df.shape)
print(df.columns.tolist())

(10, 15)
['id', 'name', 'username', 'email', 'phone', 'website', 'address.street', 'address.suite', 'address.city', 'address.zipcode', 'address.geo.lat', 'address.geo.lng', 'company.name', 'company.catchPhrase', 'company.bs']


In [110]:
df = df.rename(columns={
    "address.street": "street",
    "address.suite": "suite",
    "address.city": "city",
    "address.zipcode": "zipcode",
    "address.geo.lat": "latitude",
    "address.geo.lng": "longitude",
    "company.name": "company_name",
    "company.catchPhrase": "company_catchphrase",
    "company.bs": "company_bs"
})

print(df.columns.tolist())

['id', 'name', 'username', 'email', 'phone', 'website', 'street', 'suite', 'city', 'zipcode', 'latitude', 'longitude', 'company_name', 'company_catchphrase', 'company_bs']


In [111]:
df["latitude"] = pd.to_numeric(df["latitude"])
df["longitude"] = pd.to_numeric(df["longitude"])

print(df[["latitude", "longitude"]].dtypes)

latitude     float64
longitude    float64
dtype: object


In [112]:
def validate_users(df):
    assert df["id"].notna().all(), "Missing user IDs"
    assert df["id"].is_unique, "Duplicate user IDs"
    assert df["email"].notna().all(), "Missing emails"
    assert df["latitude"].between(-90, 90).all(), "Invalid latitude"
    assert df["longitude"].between(-180, 180).all(), "Invalid longitude"
    return True

print(validate_users(df))

True


In [113]:
import os

os.makedirs("data/processed", exist_ok=True)

df.to_parquet(
    "data/processed/users.parquet",
    index=False
)

print(os.path.exists("data/processed/users.parquet"))

True


In [114]:
processed_df = pd.read_parquet("data/processed/users.parquet")

print(processed_df.shape)
print(processed_df.dtypes)
print(validate_users(processed_df))

(10, 15)
id                       int64
name                    object
username                object
email                   object
phone                   object
website                 object
street                  object
suite                   object
city                    object
zipcode                 object
latitude               float64
longitude              float64
company_name            object
company_catchphrase     object
company_bs              object
dtype: object
True


In [115]:
posts_url = "https://jsonplaceholder.typicode.com/posts"

posts_response = requests.get(posts_url)

print(posts_response.status_code)

200


In [116]:
posts_data = posts_response.json()

print(type(posts_data))
print(len(posts_data))

<class 'list'>
100


In [117]:
print(posts_data[0])

{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


In [118]:
posts_df = pd.DataFrame(posts_data)

print(posts_df.shape)
print(posts_df.columns.tolist())
print(posts_df.head())

(100, 4)
['userId', 'id', 'title', 'body']
   userId  id                                              title  \
0       1   1  sunt aut facere repellat provident occaecati e...   
1       1   2                                       qui est esse   
2       1   3  ea molestias quasi exercitationem repellat qui...   
3       1   4                               eum et est occaecati   
4       1   5                                 nesciunt quas odio   

                                                body  
0  quia et suscipit\nsuscipit recusandae consequu...  
1  est rerum tempore vitae\nsequi sint nihil repr...  
2  et iusto sed quo iure\nvoluptatem occaecati om...  
3  ullam et saepe reiciendis voluptatem adipisci\...  
4  repudiandae veniam quaerat sunt sed\nalias aut...  


In [119]:
print(posts_df.info())

print("\nMissing values:")
print(posts_df.isnull().sum())

print("\nDuplicate post IDs:")
print(posts_df["id"].duplicated().sum())

print("\nUnique users referenced:")
print(posts_df["userId"].nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   userId  100 non-null    int64 
 1   id      100 non-null    int64 
 2   title   100 non-null    object
 3   body    100 non-null    object
dtypes: int64(2), object(2)
memory usage: 3.3+ KB
None

Missing values:
userId    0
id        0
title     0
body      0
dtype: int64

Duplicate post IDs:
0

Unique users referenced:
10


In [120]:
invalid_user_ids = posts_df[
    ~posts_df["userId"].isin(df["id"])
]

print("Invalid user IDs:", len(invalid_user_ids))

Invalid user IDs: 0


In [121]:
import json
import os

os.makedirs("data/raw", exist_ok=True)

with open("data/raw/posts.json", "w") as f:
    json.dump(posts_data, f, indent=2)

print(os.path.exists("data/raw/posts.json"))

True


In [122]:
os.makedirs("data/processed", exist_ok=True)

posts_df.to_parquet(
    "data/processed/posts.parquet",
    index=False
)

print(os.path.exists("data/processed/posts.parquet"))

True


In [123]:
processed_posts_df = pd.read_parquet(
    "data/processed/posts.parquet"
)

print(processed_posts_df.shape)
print(processed_posts_df.dtypes)
print(processed_posts_df.head(3))

(100, 4)
userId     int64
id         int64
title     object
body      object
dtype: object
   userId  id                                              title  \
0       1   1  sunt aut facere repellat provident occaecati e...   
1       1   2                                       qui est esse   
2       1   3  ea molestias quasi exercitationem repellat qui...   

                                                body  
0  quia et suscipit\nsuscipit recusandae consequu...  
1  est rerum tempore vitae\nsequi sint nihil repr...  
2  et iusto sed quo iure\nvoluptatem occaecati om...  


In [124]:
import duckdb

con = duckdb.connect()

print(
    con.execute("""
        SELECT COUNT(*)
        FROM 'data/processed/users.parquet'
    """).fetchone()[0]
)

print(
    con.execute("""
        SELECT COUNT(*)
        FROM 'data/processed/posts.parquet'
    """).fetchone()[0]
)

10
100


In [125]:
joined_df = con.execute("""
    SELECT
        p.id AS post_id,
        p.title,
        u.id AS user_id,
        u.name,
        u.email,
        u.city
    FROM 'data/processed/posts.parquet' AS p
    JOIN 'data/processed/users.parquet' AS u
        ON p.userId = u.id
""").fetchdf()

print(joined_df.shape)
print(joined_df.head())

(100, 6)
   post_id                                              title  user_id  \
0        1  sunt aut facere repellat provident occaecati e...        1   
1        2                                       qui est esse        1   
2        3  ea molestias quasi exercitationem repellat qui...        1   
3        4                               eum et est occaecati        1   
4        5                                 nesciunt quas odio        1   

            name              email         city  
0  Leanne Graham  Sincere@april.biz  Gwenborough  
1  Leanne Graham  Sincere@april.biz  Gwenborough  
2  Leanne Graham  Sincere@april.biz  Gwenborough  
3  Leanne Graham  Sincere@april.biz  Gwenborough  
4  Leanne Graham  Sincere@april.biz  Gwenborough  


In [126]:
post_counts = con.execute("""
    SELECT
        u.id AS user_id,
        u.name,
        COUNT(p.id) AS post_count
    FROM 'data/processed/users.parquet' AS u
    LEFT JOIN 'data/processed/posts.parquet' AS p
        ON u.id = p.userId
    GROUP BY
        u.id,
        u.name
    ORDER BY post_count DESC
""").fetchdf()

print(post_counts)

   user_id                      name  post_count
0        3          Clementine Bauch          10
1        5          Chelsey Dietrich          10
2        1             Leanne Graham          10
3        2              Ervin Howell          10
4        4          Patricia Lebsack          10
5        8  Nicholas Runolfsdottir V          10
6        6      Mrs. Dennis Schulist          10
7        7           Kurtis Weissnat          10
8        9           Glenna Reichert          10
9       10        Clementina DuBuque          10


In [127]:
title_stats = con.execute("""
    SELECT
        u.id AS user_id,
        u.name,
        ROUND(AVG(LENGTH(p.title)), 2) AS avg_title_length
    FROM 'data/processed/users.parquet' AS u
    JOIN 'data/processed/posts.parquet' AS p
        ON u.id = p.userId
    GROUP BY
        u.id,
        u.name
    ORDER BY avg_title_length DESC
""").fetchdf()

print(title_stats)

   user_id                      name  avg_title_length
0        5          Chelsey Dietrich              48.4
1        8  Nicholas Runolfsdottir V              43.6
2        9           Glenna Reichert              42.3
3        6      Mrs. Dennis Schulist              42.1
4        2              Ervin Howell              40.0
5        7           Kurtis Weissnat              37.4
6        3          Clementine Bauch              37.1
7       10        Clementina DuBuque              35.4
8        4          Patricia Lebsack              35.1
9        1             Leanne Graham              33.8


In [128]:
con.execute("""
    COPY (
        SELECT
            p.id AS post_id,
            p.userId AS user_id,
            u.name AS user_name,
            u.email,
            u.city,
            p.title,
            p.body
        FROM 'data/processed/posts.parquet' AS p
        JOIN 'data/processed/users.parquet' AS u
            ON p.userId = u.id
    )
    TO 'data/processed/user_posts.parquet'
    (FORMAT PARQUET)
""")

print(os.path.exists("data/processed/user_posts.parquet"))

True
